# d=96 Convergence Extension — is the "no ε floor" result real convergence or undertraining?

Fully standalone — the only external dependency is the **checkpoint and data
pool files on disk** from the original Experiment 6 big-pool run:
`./exp6_bigpool_h4/shared_d96_h4_seed{0,7}.pt` and
`./exp6_bigpool_h4/data_shared_seed{1,7}.pt`. Run this from the same working
directory as that run, or copy those four files over first. Everything else
(grammar, model, eval functions) is redefined here from scratch so this
notebook doesn't depend on any other notebook's live state.

**Why this exists:** at the 90K-step checkpoint, d=96 train CE was 0.7972
against an entropy floor of 0.7545 — not saturated. The headline "CE gap
fades to ≈0 by d=96, no ε floor" result assumes that gap is measured at
convergence. This notebook extends two seeds by another 90K steps (180K
total) on the exact same data and checks whether train CE keeps closing
toward the floor, and whether the CE gap moves while it does.


## 0. Device, paths, and config (must match the original run exactly)

In [1]:
import os, math, random, subprocess, sys, time
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")

CK2 = "./exp6_bigpool_h4"
assert os.path.isdir(CK2), f"{CK2} not found — copy the original big-pool checkpoint dir here first"

def bp_shared_path(d, s): return os.path.join(CK2, f"shared_d{d}_h4_seed{s}.pt")
def bp_data(name): return os.path.join(CK2, f"data_{name}.pt")

# ---- config, must match Experiment 6 exactly ----
LEVELS = 6
NT_SIZE = 4
TERM_SIZE = 4
MIN_DEGREE = 2
MAX_DEGREE = 4
RULE_LEN_CHOICES = [2, 3]
LANGS = {"L_A": set(), "L_B": {4, 5}, "L_C": set(range(1, LEVELS + 1))}
LANG_NAMES = list(LANGS.keys())
MIX = {"L_A": 0.70, "L_B": 0.15, "L_C": 0.15}
VOCAB_SIZE = TERM_SIZE + len(LANG_NAMES)
BOS_OFFSET = TERM_SIZE
BOS = {name: BOS_OFFSET + i for i, name in enumerate(LANG_NAMES)}
SEQ_LEN_CHECK = 256
N_HEAD = 4
FIXED_BUDGET_SWEEP = 90_000

for seed in (0, 7):
    p = bp_shared_path(96, seed)
    assert os.path.exists(p), f"missing checkpoint: {p}"
ds_map = {0: 1, 7: 7}
for ds in set(ds_map.values()):
    p = bp_data(f"shared_seed{ds}")
    assert os.path.exists(p), f"missing data pool: {p}"
print("all required files present")


all required files present


## 1. Grammar (same seed as the original — reproduces the identical grammar object)

In [2]:
def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))
    from collections import defaultdict
    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = terminals if child_level == LEVELS + 1 else symbols_by_level[child_level]
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set(); tries = 0
            while len(rules[sym]) < degree and tries < 200:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen: continue
                seen.add(rhs); rules[sym].append(rhs)
    return rules, symbols_by_level

def sample_tree(rules, rng, symbol=(1, 0), level=1):
    rhs = rng.choice(rules[symbol])
    children = []
    for child_sym in rhs:
        children.append(child_sym if level == LEVELS else sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)

def linearize(node, reverse_levels):
    if isinstance(node, int):
        return [node]
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens = []
    for c in order:
        tokens.extend(linearize(c, reverse_levels))
    return tokens

check_rules, check_symbols_by_level = build_grammar(seed=0)   # same seed as original Exp 6

def make_shared_tree_mix(rules, n_trees, seed, seq_len, mix=MIX):
    rng = random.Random(seed)
    stream = []
    for _ in range(n_trees):
        lang = rng.choices(LANG_NAMES, weights=[mix[n] for n in LANG_NAMES])[0]
        tree = sample_tree(rules, rng)
        stream.append(BOS[lang]); stream.extend(linearize(tree, LANGS[lang]))
    stream = torch.tensor(stream, dtype=torch.long)
    n_chunks = len(stream) // (seq_len + 1)
    return stream[: n_chunks * (seq_len + 1)].view(n_chunks, seq_len + 1)

def per_language_loss(model, data, lang_name, device=DEVICE):
    mask = data[:, 0] == BOS[lang_name]
    subset = data[mask]
    if subset.shape[0] == 0: return None
    x, y = subset[:, :-1].to(device), subset[:, 1:].to(device)
    with torch.no_grad(), torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                                          dtype=torch.bfloat16):
        _, loss = model(x, y)
    return loss.item()

def _walk(rules, rng, symbol, level):
    rhs_list = rules[symbol]
    rhs = rng.choice(rhs_list)
    nats = -math.log(rhs_list.count(rhs) / len(rhs_list))
    if level == LEVELS:
        return nats, len(rhs)
    leaves = 0
    for child in rhs:
        n, l = _walk(rules, rng, child, level + 1)
        nats += n; leaves += l
    return nats, leaves

def entropy_per_token(rules, n_trees=20000, seed=0):
    rng = random.Random(seed)
    tot_n = tot_l = 0
    for _ in range(n_trees):
        n, l = _walk(rules, rng, (1, 0), 1)
        tot_n += n; tot_l += l
    return tot_n / tot_l, tot_l / n_trees

H_sh, leaves_sh = entropy_per_token(check_rules)
print(f"grammar entropy H_sh = {H_sh:.4f} nats/token ({leaves_sh:.0f} leaves/tree)")


grammar entropy H_sh = 0.7545 nats/token (238 leaves/tree)


## 2. Model definition (must match the checkpoint's architecture exactly)

In [3]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0
        assert (n_embd // n_head) % 2 == 0
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks: x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

def load_shared_ckpt(path, d_model, seq_len, vocab_size=VOCAB_SIZE, n_layer=12, n_head=N_HEAD, device=DEVICE):
    ck = torch.load(path, map_location=device, weights_only=False)
    m = TinyRotaryGPT(vocab_size, n_embd=d_model, n_layer=n_layer, n_head=n_head,
                       max_seq_len=seq_len + 1).to(device)
    m.load_state_dict(ck["model_state"]); m.eval()
    return m

@torch.no_grad()
def mean_ce(model, data, bs=256):
    model.eval(); tot = n = 0
    for i in range(0, data.shape[0], bs):
        b = data[i:i + bs].to(DEVICE)
        x, y = b[:, :-1], b[:, 1:]
        with torch.autocast(device_type="cuda" if DEVICE.type == "cuda" else "cpu", dtype=torch.bfloat16):
            _, loss = model(x, y)
        tot += loss.item() * x.numel(); n += x.numel()
    return tot / n


## 3. Resume-capable training worker

Writes a standalone `.py` script (needed because it runs as a subprocess,
separate from this notebook's process) that loads an existing checkpoint and
continues training on the same data for `--extra_steps` more steps, saving
to a NEW path so the original 90K-step checkpoint is untouched.

In [4]:
worker_src = '''
import argparse, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0
        assert (n_embd // n_head) % 2 == 0
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks: x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--d", type=int, required=True)
    ap.add_argument("--seed", type=int, required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--resume_from", required=True)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--vocab", type=int, required=True)
    ap.add_argument("--seq_len", type=int, required=True)
    ap.add_argument("--extra_steps", type=int, required=True)
    ap.add_argument("--n_head", type=int, default=4)
    ap.add_argument("--n_layer", type=int, default=12)
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=3e-3)
    a = ap.parse_args()

    torch.set_num_threads(1)
    device = torch.device(a.device)
    torch.manual_seed(a.seed)
    data = torch.load(a.data).to(device)

    ck = torch.load(a.resume_from, map_location=device, weights_only=False)
    model = TinyRotaryGPT(a.vocab, n_embd=a.d, n_layer=a.n_layer, n_head=a.n_head,
                          max_seq_len=a.seq_len + 1).to(device)
    model.load_state_dict(ck["model_state"])
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, fused=True)
    if "opt_state" in ck:
        opt.load_state_dict(ck["opt_state"])
    prior_step = ck.get("step", 0)
    prior_losses = ck.get("losses", [])

    n = data.shape[0]
    buf = torch.zeros(a.extra_steps, device=device)
    t0 = time.time()
    for step in range(a.extra_steps):
        idx = torch.randint(0, n, (a.batch,), device=device)
        batch = data[idx]
        x, y = batch[:, :-1], batch[:, 1:]
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = model(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
        buf[step] = loss.detach().float()
        if (step + 1) % 5000 == 0:
            print(f"extra step {step+1:>7d} (total {prior_step + step + 1}): "
                  f"mean loss over last 5000 = {buf[step-4999:step+1].mean().item():.4f}"
                  f"  ({(step + 1) / (time.time() - t0):.0f} it/s)", flush=True)
        if (step + 1) % 10000 == 0 or step == a.extra_steps - 1:
            tmp = a.ckpt + ".tmp"
            torch.save({"model_state": model.state_dict(), "opt_state": opt.state_dict(),
                        "step": prior_step + step + 1,
                        "losses": prior_losses + buf[:step + 1].tolist()}, tmp)
            os.replace(tmp, a.ckpt)

if __name__ == "__main__":
    main()
'''
with open("train_worker_resume.py", "w") as f:
    f.write(worker_src)
print("wrote train_worker_resume.py")


wrote train_worker_resume.py


## 4. Launch: extend d=96, seeds 0 and 7, by 90K more steps

In [5]:
def run_parallel(jobs, max_par=2):
    running = []
    for name, cmd in jobs:
        print(f"launching {name}")
        running.append((name, subprocess.Popen(cmd)))
        while len(running) >= max_par:
            for n, p in running:
                if p.poll() is not None:
                    running.remove((n, p)); break
            else:
                time.sleep(5)
    for name, p in running:
        p.wait()
        print(f"{name} done, exit {p.returncode}")

def ckpt_step(path):
    if not os.path.exists(path): return None
    try:
        return torch.load(path, map_location="cpu", weights_only=False).get("step")
    except Exception:
        return None

EXT_STEPS = 90_000
EXT_SEEDS = [0, 7]

def ext_path(seed):
    return os.path.join(CK2, f"shared_d96_h4_seed{seed}_ext180k.pt")

jobs = []
for seed in EXT_SEEDS:
    target_step = FIXED_BUDGET_SWEEP + EXT_STEPS
    if ckpt_step(ext_path(seed)) == target_step:
        continue
    data_seed = ds_map[seed]
    jobs.append((f"ext_d96_seed{seed}",
                 [sys.executable, "train_worker_resume.py",
                  "--d", "96", "--seed", str(seed),
                  "--data", bp_data(f"shared_seed{data_seed}"),
                  "--resume_from", bp_shared_path(96, seed),
                  "--ckpt", ext_path(seed),
                  "--vocab", str(VOCAB_SIZE), "--seq_len", str(SEQ_LEN_CHECK),
                  "--extra_steps", str(EXT_STEPS), "--n_head", str(N_HEAD), "--device", str(DEVICE)]))

print(f"{len(jobs)} convergence-extension runs to do")
run_parallel(jobs, max_par=2)

missing = [s for s in EXT_SEEDS if ckpt_step(ext_path(s)) != FIXED_BUDGET_SWEEP + EXT_STEPS]
assert not missing, f"extension incomplete for seeds: {missing}"
print("d=96 convergence extension complete")


2 convergence-extension runs to do
launching ext_d96_seed0
launching ext_d96_seed7
extra step    5000 (total 95000): mean loss over last 5000 = 0.7981  (33 it/s)
extra step    5000 (total 95000): mean loss over last 5000 = 0.7971  (30 it/s)
extra step   10000 (total 100000): mean loss over last 5000 = 0.7978  (33 it/s)
extra step   10000 (total 100000): mean loss over last 5000 = 0.7968  (30 it/s)
extra step   15000 (total 105000): mean loss over last 5000 = 0.7973  (33 it/s)
extra step   15000 (total 105000): mean loss over last 5000 = 0.7966  (31 it/s)
extra step   20000 (total 110000): mean loss over last 5000 = 0.7970  (33 it/s)
extra step   20000 (total 110000): mean loss over last 5000 = 0.7965  (31 it/s)
extra step   25000 (total 115000): mean loss over last 5000 = 0.7968  (32 it/s)
extra step   25000 (total 115000): mean loss over last 5000 = 0.7961  (31 it/s)
extra step   30000 (total 120000): mean loss over last 5000 = 0.7964  (32 it/s)
extra step   30000 (total 120000): mean

## 5. Readout: did train CE keep closing toward the entropy floor, and did the CE gap move?

In [6]:
fresh_mix_ext = make_shared_tree_mix(check_rules, n_trees=20_000, seed=12345, seq_len=SEQ_LEN_CHECK, mix=MIX)

print(f"entropy floor H_sh = {H_sh:.4f}")
print(f"{'seed':>4s} | {'train CE @90K':>13s} {'train CE @180K':>14s} | {'CE-gap B @90K':>13s} {'CE-gap B @180K':>14s}")
for seed in EXT_SEEDS:
    tr_90k = float(sum(torch.load(bp_shared_path(96, seed), map_location="cpu", weights_only=False)["losses"][-1000:]) / 1000)
    tr_180k = float(sum(torch.load(ext_path(seed), map_location="cpu", weights_only=False)["losses"][-1000:]) / 1000)
    m_90k = load_shared_ckpt(bp_shared_path(96, seed), 96, SEQ_LEN_CHECK)
    m_180k = load_shared_ckpt(ext_path(seed), 96, SEQ_LEN_CHECK)
    gap_90k = per_language_loss(m_90k, fresh_mix_ext, "L_B") - mean_ce(m_90k, fresh_mix_ext)
    gap_180k = per_language_loss(m_180k, fresh_mix_ext, "L_B") - mean_ce(m_180k, fresh_mix_ext)
    del m_90k, m_180k; torch.cuda.empty_cache()
    print(f"{seed:>4d} | {tr_90k:13.4f} {tr_180k:14.4f} | {gap_90k:13.4f} {gap_180k:14.4f}")


entropy floor H_sh = 0.7545
seed | train CE @90K train CE @180K | CE-gap B @90K CE-gap B @180K
   0 |        0.7973         0.7941 |       -0.0085        -0.0065
   7 |        0.7985         0.7937 |       -0.0092        -0.0071


## 6. Reading the result

- Train CE moving closer to `H_sh` at 180K, with the CE gap staying flat at
  ≈0 → the "no ε floor at d=96" conclusion holds; the model was underfit but
  the gap was already at its converged value.
- Train CE moving toward the floor AND the CE gap growing or shrinking
  meaningfully → the original 90K-step reading was premature. Re-run the
  full d=96 sweep (all 3 seeds) at 180K before drawing conclusions from it,
  and reconsider whether other d values in the sweep need the same check.
